# 다층 퍼셉트론으로 클래스 모델 구성 및 하이퍼파라미터 최적화

- 목차
1. 라이브러리 import
2. 데이터 준비
    - 학습/검증/테스트 데이터 분리
    - 데이터 표준화
    - Tensor 형태로 데이터 변환
3. 미니배치 준비
4. 모델 준비
5. 학습 및 평가 함수 (+Early stopping) 
6. Optuna 활용하여 하이퍼파라미터 최적화
7. 최적의 파라미터로 재학습
8. 시각화   

1. 라이브러리 import

In [118]:
# 기본 라이브러리
import numpy as np
import matplotlib.pyplot as plt

# 데이터, 데이터 분리, 스케일링, 지표
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay

# 텐서 조작 및 Pytorch 기본, 레이어 구성, 최적화함수, 배치데이터셋 구성
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader


# 기타 라이브러리
import optuna                       # 최적화
import warnings                     # 경고 메시지 제어
warnings.filterwarnings('ignore')   # 경고 메시지 무시
from tqdm import tqdm               # 프로그래스(진행률) 바



#한글 폰트 설정 (Mac은 AppleGothic) 
plt.rcParams["font.family"] = "Malgun Gothic"

#마이너스 기호 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False

2. 데이터 준비

In [119]:
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

print(X.shape , y.shape)

(569, 30) (569,)


In [120]:
# Test 데이터 분리

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size = 0.2,     # 80:20
    random_state = 42,
    stratify = y         # 클래스 비율을 유지하면서 분할
)

In [121]:
# Train/Valid 데이터 분리 (하이퍼 파라미터 최적화용)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size = 0.25,       # 75: 25
    random_state = 42,      
    stratify = y_trainval   # y_Trainval의 클래스 비율 유지하면서 분할
)

print(X_train.shape, X_val.shape, X_test.shape) # 최종 비율 6:2:2

(341, 30) (114, 30) (114, 30)


In [122]:
# 입력 데이터 표준화(평균 0, 표준편차 1로 X데이터 스케일 조정)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(X_train.shape, X_val.shape, X_test.shape)

(341, 30) (114, 30) (114, 30)


In [123]:
# X_train, y_train, X_val, y_val, X_test, y_test 데이터 Pytorch Tensor 형태로 변환 (데이터 타입은 float)
# 하고 y 데이터는 2차원 형태로 맞춰 딥러닝 모델에 사용가능하도록 맞추기

X_train_t = torch.tensor(X_train, dtype = torch.float32)
y_train_t = torch.FloatTensor(y_train).reshape(-1,1)
X_val_t = torch.tensor(X_val).float()
y_val_t = torch.tensor(y_val, dtype = torch.float32).view(-1,1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).reshape(-1,1)

print(X_train_t.size(), y_train_t.size(), X_val_t.size(), y_val_t.size(), X_test_t.size(), y_test_t.size())



torch.Size([341, 30]) torch.Size([341, 1]) torch.Size([114, 30]) torch.Size([114, 1]) torch.Size([114, 30]) torch.Size([114, 1])


3. 미니배치 준비

In [124]:
# 미니배치 : 학습/검증/테스트용 DataLoader를 생성하는 함수
def make_loaders(batch_size = 64):
    train_loader = DataLoader(
        TensorDataset(X_train_t, y_train_t),   # 학습용 입력(X)와 정답(y)를 묶어 전달
        batch_size = batch_size,               # 미니배치 크기
        shuffle = True                         # 학습용 데이터는 섞어서 일반화 성능 향상
    )

    val_loader = DataLoader(
        TensorDataset(X_val_t, y_val_t),
        batch_size = batch_size, 
        shuffle = False                        # 검증용 데이터는 순서 고정
    ) 
    test_loader = DataLoader(
        TensorDataset(X_test_t, y_test_t),
        batch_size = batch_size,
        shuffle = False                        # 테스트 데이터도 순서 고정
    )

    return train_loader, val_loader, test_loader   # 3종류의 DataLoader 반환

4. 모델 준비

In [125]:
# 다층 퍼셉트론 이진 분류 모델

class MLP(nn.Module):

    def __init__(self, input_dim, hidden_dim1 = 64, hidden_dim2 = 32, dropout_p = 0.0):
        super().__init__()                           # nn.Module 초기화

        self.fc1 = nn.Linear(input_dim, hidden_dim1)   # 입력 -> 첫번째 은닉층
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2) # 첫 번째 은닉층 -> 두 번째 은닉층
        self.fc_out = nn.Linear(hidden_dim2,1)         # 출력층 : 두 번째 은닉층 -> logit 값 1개 출력

        self.act = nn.ReLU()                           # 활성화 함수
        self.dropout = nn.Dropout(dropout_p) if dropout_p > 0 else nn.Identity()  # Dropout 사용 또는 없을시 통과

    def forward(self, x):
        x = self.act(self.fc1(x))  # 첫 번째 은닉층 + ReLU
        x = self.dropout(x)        # Dropout 적용 (선택)
        x = self.act(self.fc2(x))  # 두 번째 은닉층 + ReLU
        x = self.dropout(x)        # Dropout 적용 (선택)
        x = self.fc_out(x)         # 출력층 (logits)

        return x # (batch_size, 1) 형태로 출력


5. 학습 및 평가 함수

In [126]:
criterion = nn.BCEWithLogitsLoss()

# 학습 함수
def train_one_epoch(model, loader, optimizer):
    model.train()                       # 학습 모드로 전환
    losses = []                         # 배치별 loss를 전환할 리스트

    for xb, yb in loader:               # 미니배치 단위로 (입력, 정답) 반복
        logits = model(xb)              # 순전파 : 모델 출력(logit) 계산
        loss = criterion(logits, yb)    # 예측(logit)과 정답(0/1)로 BCE 손실 계산

        optimizer.zero_grad()           # 이전 step의 기울기 초기화
        loss.backward()                 # 약잔피 : 현재 loss 기준으로 기울기 계산
        optimizer.step()                # 계산된 기울기로 파라미터(가중치/편향) 업데이트

        losses.append(loss.item())      # 파이썬 숫자 형태로 loss 값 저장

    return float(np.mean(losses))       # 1 epoch 동안 평균 train loss 반환

In [127]:
# 평가 함수
@torch.no_grad() # 데코레이터 방식 (함수 안에서 with torch.no_grad(): 구문 사용과 같음)
def evaluate(model, loader):
    model.eval()               # 평가 모드로 전환 (dropout 비활성화, batchNorm은 고정 통계 사용: 비활성화)
    losses = []                # 배치별 loss
    y_true_all = []            # 실제 정답
    y_prob_all = []            # 예측 확률

    # 미니배치 단위로 반복
    for xb, yb in loader:
        logits = model(xb)             # 순전파 : 모델 출력(logit) 계산
        loss = criterion(logits, yb)   # 검증/테스트 loss 계산
        prob = torch.sigmoid(logits)   # logit -> 확률(0~1) 변환 (시각화/평가지표)

        losses.append(loss.item())             # loss 기록
        y_true_all.append(yb.cpu().numpy())    # 정답을 numpy 형태로 변환
        y_prob_all.append(prob.cpu().numpy())  # 예측 확률도 numpy 형태로 변환

    y_true = np.vstack(y_true_all).ravel()     # (N,1) -> (N,) 형태로 변환
    y_prob = np.vstack(y_prob_all).ravel()     # (N,1) -> (N,) 형태로 변환
    y_pred = (y_prob > 0.5).astype(int)        # 임계값 0.5 기준으로 클래스 (0/1) 결정

    acc = accuracy_score(y_true, y_pred)       # 정확도 계산 
    auc = roc_auc_score(y_true, y_prob)        # ROC-AUC 계산 (확률 기반)
    return float(np.mean(losses)), acc, auc, y_true, y_prob  # 평균 Loss, acc, auc, 정답/확률 반환

```

np.vstack(y_true_all).ravel()
y_true_all = [
    array([[1], [0], [1], [0], ...]) # 첫 번째 배치
    array([[0], [1], [0], [1], ...]) # 두 번쨰 배치
]

np.vstack() : 세로로 붙이기 (배치들을 하나로 이어 붙여줌)

array([
    [1],
    [0],
    [1],
    [0],
    ...
])

ravel() : 배열들을 1차원으로 평탄화 (flatten)
(N,1) -> (N, )
array([1, 0, 1, 0, 0, 1, ...])

np.vstack(y_true_all).reshape(-1) 해도 동일한 효과
np.vstack(y_true_all).flatten() 해도 동일한 효과

이렇게 사용하는 이유 : accuracy-score나 roc_auc_score 등 각종 평가지표들이 1차원 배열을 매개값으로 요구함.
```

In [128]:
# Early Stopping : 검증시 손실이 더 이상 개선되지 않으면 학습을 조기에 중단하는 클래스
class EarlyStopping:

    def __init__(self, patience = 20, min_delta = 0.0):
        self.patience = patience                   # 성능이 개선이 없어도 허용할 epoch
        self.min_delta = min_delta                 # 개선으로 인정할 최소 손실 감소량
        self.best = float("inf")                   # 최소 검증 손실
        self.counter = 0                           # 연속으로 개선되지 않은 epoch 수
        self.stop = False                          # 학습 중단 여부 플래그

    def step(self, val_loss):
        # 최대 손실에서 최소 손실 감소량만큼 뺀 것보다 더 손실이 적어지면 (개선되면)
        if val_loss < self.best - self.min_delta:
            self.best = val_loss               # 해당 loss 기록
            self.counter = 0                       # 카운터 초기화
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True                   # 학습 중단 신호

In [129]:
# 모델 학습, 검증 손실 기록, Early Stopping으로 과적합 방지하는 함수
def fit(model, train_loader, val_loader, lr = 1e-3, n_epochs =200, weight_decay = 0.0):
    optimizer = optim.Adam(
        model.parameters(),
        lr = lr,
        weight_decay = weight_decay
    )

    hist = {"train_loss": [], "val_loss": [], "val_acc": [], "val_auc": []}
    best_val = np.inf
    best_state = None

    early = EarlyStopping(patience = 20)

    for epoch in range(1, n_epochs + 1):
        tr_loss = train_one_epoch(model, train_loader, optimizer)
        va_loss, va_acc, va_auc, _, _ = evaluate(model, val_loader)

        hist["train_loss"].append(tr_loss)
        hist["val_loss"].append(va_loss)
        hist["val_acc"].append(va_acc)
        hist["val_auc"].append(va_auc)

        if va_loss < best_val:
            best_val = va_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        early.step(va_loss)
        if early.stop:
            print(f"Epochs {epoch}에서 종료됩니다.")
            break    

    if best_state is not None:
        model.load_state_dict(best_state) 

    return hist   


    

6. Optuna 활용하여 하이퍼파라미터 최적화

In [ ]:
def objective(trial):
    hidden_dim1 = trial.suggest_categorical("hidden_dim1", [32,64,128,256]) # 은닉 1층 노드
    hidden_dim2 = trial.suggest_categorical("hidden_dim2", [16,32,64,128])  # 은닉 2층 노드
    dropout_p = trial.suggest_float("dropout_p", 0.0, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log = True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

    batch_size = trial.suggest_categorical("batch_size", [32,64,128])
    train_loader, val_loader, _ = make_loaders(batch_size = batch_size)

    model = MLP(
        input_dim = X_train.shape[-1],
        hidden_dim1 = hidden_dim1,
        hidden_dim2 = hidden_dim2,
        dropout_p = dropout_p
    )

    hist = fit(
        model, train_loader, val_loader,
        lr = lr,
        n_epochs = 150,
        weight_decay = weight_decay
    )

    best_val_auc = float(np.max(hist["val_auc"]))
    return best_val_auc   # Optuna가 최대화할 목적 함수 값

study = optuna.create_study(direction = "maximize")  # Optuna Study 객체
study.optimize(objective, n_trials = 20)             # objective 함수 20회 실행하며 탐색

print(f"옵튜나 최고 성능 조합 : {study.best_params}")  # 최적의 하이퍼파라미터 조합
print(f"옵튜나 최고 성능의 val_auc : {study}")         # 해당 파라미터에서의 최고 성능 AUC



[I 2026-07-28 09:56:27,876] A new study created in memory with name: no-name-88d17058-76df-4892-90de-108c4eeca243
[I 2026-07-28 09:56:29,151] Trial 0 finished with value: 0.9980347199475925 and parameters: {'hidden_dim1': 32, 'hidden_dim2': 32, 'dropout_p': 0.09088365592456887, 'lr': 0.0024855132536108692, 'weight_decay': 4.262253774582673e-05, 'batch_size': 64}. Best is trial 0 with value: 0.9980347199475925.


Epochs 75에서 종료됩니다.


[I 2026-07-28 09:56:30,476] Trial 1 finished with value: 0.99737962659679 and parameters: {'hidden_dim1': 32, 'hidden_dim2': 16, 'dropout_p': 0.43280495327452534, 'lr': 0.002974390533869552, 'weight_decay': 1.1262073395899677e-05, 'batch_size': 64}. Best is trial 0 with value: 0.9980347199475925.


Epochs 80에서 종료됩니다.


[I 2026-07-28 09:56:32,269] Trial 2 finished with value: 0.9977071732721913 and parameters: {'hidden_dim1': 32, 'hidden_dim2': 32, 'dropout_p': 0.10670942208273193, 'lr': 0.0011651687922062197, 'weight_decay': 1.3195393770349967e-06, 'batch_size': 32}. Best is trial 0 with value: 0.9980347199475925.


...

Epochs 29에서 종료됩니다.


[I 2026-07-28 09:56:47,530] Trial 18 finished with value: 0.9977071732721913 and parameters: {'hidden_dim1': 256, 'hidden_dim2': 128, 'dropout_p': 0.3171346895872707, 'lr': 0.000377742146671501, 'weight_decay': 1.9043052857299972e-05, 'batch_size': 32}. Best is trial 5 with value: 0.9983622666229938.


Epochs 51에서 종료됩니다.


[I 2026-07-28 09:56:48,198] Trial 19 finished with value: 0.9983622666229938 and parameters: {'hidden_dim1': 256, 'hidden_dim2': 128, 'dropout_p': 0.22629918824365422, 'lr': 0.0016866171339287724, 'weight_decay': 0.00015754409632690418, 'batch_size': 32}. Best is trial 5 with value: 0.9983622666229938.


Epochs 34에서 종료됩니다.
옵튜나 최고 성능 조합 : {'hidden_dim1': 256, 'hidden_dim2': 128, 'dropout_p': 0.024022896147839312, 'lr': 0.0014911348407969764, 'weight_decay': 2.1386400799893604e-05, 'batch_size': 32}
옵튜나 최고 성능의 val_auc : <optuna.study.study.Study object at 0x00000227D5840800>


7. 최적의 하이퍼파라미터로 재학습

In [137]:
best = study.best_params

train_loader, val_loder, test_loader = make_loaders(batch_size = best['batch_size'])

best_model = MLP(
    input_dim = X_train.shape[-1],
    hidden_dim1 = best['hidden_dim1'],
    hidden_dim2 = best['hidden_dim2'],
    dropout_p = best['dropout_p']
)                                

history = fit(
    best_model,
    train_loader,
    val_loader,
    lr = best['lr'],

    n_epochs = 1000,
    weight_decay = best['wight_dacay']
)

test_loss, test_acc, test_auc, y_true,y_prob = evaluate(best_model, test__loader)



print(f"Test 기준 최고 정확도 : {test_acc:.4f}, 최저 loss : {test_loss}, auc : {test_auc:.4f}")
                                                                       

NameError: name 'val_loader' is not defined

8. 시각화

In [ ]:
#Train / Valid Loss 학습 곡선, Val Auc
plt.figure(figsize =(8,5))

손실 곡선 : 학습 손실은 줄어드는데 검증 손실을ㄴ 줄어들지 않으면 과적합 신호

AUC곡선: 모델의 전반적인 분류 능력이 어떻게 변하는지확인
AAUC 가 꾸준히 상승하면 클래스간 구분 능력이 활성화돠=ㅓ어있


두 클래스가 겹치는 부붑ㄴ은 약간 존재하긴 하지만 그렇게 크지 않음
 분류 경계가 명확, 모댈 확신 높은편

1.혼자서 공부하면서 적용해볼만한 사함
1. 은닉층 증축 설계 % 하이퍼파라미터 최적화
2. 최적화 함수 변경 (Adam -> AdamW, NAdam, RobustScaler)
3. 스케일러 변경 (StandardScaker -> MinMaxScaler, RobustSclaer)
4. 데이터 불균형시 SMOTE
5.과적합 발생시

- Dropout / BatchNorm
- Scheduler : lr을 점진적으로 조절 ()